# Потокенная разметка

In [5]:
from pipeline_func import tokenize

/content/KazRusCSW/pipeline_func.py:8: SyntaxWarning: invalid escape sequence '\['
  patt = f'\[[A-Z]+\]|\s|\\\\n|\d+|[{punctuation},]|\w+|.'
/content/KazRusCSW/pipeline_func.py:8: SyntaxWarning: invalid escape sequence '\w'
  patt = f'\[[A-Z]+\]|\s|\\\\n|\d+|[{punctuation},]|\w+|.'


In [9]:
from datasets import Dataset

# загружаем отфильтрованные комментарии
data = Dataset.from_csv('data/all_data_src__filtered.csv')
data

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['Unnamed: 0', 'uuid', 'clean_comment_text', 'source', 'ts', 'label', '__proba'],
    num_rows: 1009159
})

In [37]:
def extract_tokens(chunk, text_col: str='clean_comment_text') -> dict:
    """
    К батчу датасета применяется функция pipeline_func.tokenize: 
    из текста извлекаются отдельные токены (слова, числа, пунктуация -- все, кроме пробелов)
    и позиция начала и конца для каждого токена.

    Arguments:
    chunk: батч датасета
    text_col: название столбца, в котором содержится текст комментария
    
    Returns:
    Возвращает словарь с двумя "столбцами" -- извлеченными токенами и позициями токенов
    """
    res = list(map(tokenize, chunk[text_col]))
    return {
        'tokens': [item[0] for item in res],
        'spans':  [item[1] for item in res]
    }

In [42]:
# достаем токены и их позиции
data = data.map(extract_tokens, batched=True)

Map:   0%|          | 0/1009159 [00:00<?, ? examples/s]

## mBERT

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [44]:
import torch
from transformers import AutoModelForTokenClassification
from transformers import AutoTokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

finetuned_model_path = 'liminovna/KazRusCSW-mbert'
tokenizer = AutoTokenizer.from_pretrained(finetuned_model_path)
model = AutoModelForTokenClassification.from_pretrained(finetuned_model_path).to(device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: liminovna/KazRusCSW_mbert
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [45]:
tokenizer

BertTokenizer(name_or_path='liminovna/KazRusCSW_mbert', vocab_size=119547, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	119547: AddedToken("[MENTION]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	119548: AddedToken("[NUMBER]", rstrip=False, lstrip=False, si

In [50]:
import torch.nn.functional as F

def predict_labels(examples):
    if type(examples) == list:
        inputs_ = tokenizer(examples, return_tensors="pt", is_split_into_words=True, truncation=True, padding=True).to(device)
    else:
        inputs_ = tokenizer(examples["tokens"], return_tensors="pt", is_split_into_words=True, truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs_).logits
    # predictions = torch.argmax(logits, dim=-1)
    probs = torch.max(F.softmax(logits, dim=-1), dim=-1)

    all_words = []
    all_labels = []
    all_probas = []

    for i, word_ids in enumerate(inputs_['input_ids']):
        row_words = []
        row_labels = []
        row_probas = []
        words = tokenizer.convert_ids_to_tokens(word_ids)
        for j, (word_idx, word) in enumerate(zip(word_ids, words)):  # Set the special tokens to -100.
            if word.startswith('##'):
                # print('subtoken', word_idx, word)
                row_words[-1] = row_words[-1] + word[2:]
            elif word not in ['[SEP]', '[PAD]', '[CLS]']:
                # print('starting subtoken', word_idx, word)
                row_words.append(word)
                row_labels.append(probs.indices[i, j])
                row_probas.append(probs.values[i, j])
            else:
                # print('skipping', word_idx, word)
                pass

        # if len(examples["tokens"][i]) != len(row_words):
        #     print(f'word number mismatch! id={i}')

        all_words.append(row_words)
        all_labels.append(row_labels)
        all_probas.append(row_probas)

    inputs_["words"] = all_words
    inputs_["labels"] = all_labels
    inputs_["probas"] = all_probas

    return inputs_

In [ ]:
from datasets import Dataset
import datetime

print(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

predicted_data = data.map(predict_labels, batched=True, batch_size=4)
print(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
# 1 million rows / > 4 h

In [ ]:
predicted_data.to_pandas().to_json('data/all_data_src__annotated.jsonl', lines=True, orient='records')

## Определение CSW-текстов

In [3]:
import pandas as pd
predicted_data = pd.read_json('data/all_data_src__annotated.jsonl', lines=True)
predicted_data

,Unnamed: 0,uuid,clean_comment_text,source,ts,label,__proba,tokens,spans,words,labels,probas
0,0,EBqNfbjRnmPCYb6DbTRonQ,Респект. Чистка коррупционеров. В других регио...,telegram__egovpress,2026-05-16 22:05:08.728025,__label__rus_Cyrl,0.999533,"[Респект, ., Чистка, коррупционеров, ., В, дру...","[[0, 7], [7, 8], [9, 15], [16, 30], [30, 31], ...","[Респект, ., Чистка, коррупционеров, ., В, дру...","[0, 7, 5, 5, 7, 5, 5, 5, 5, 5, 5, 7, 5, 5, 5, ...","[0.9206345081, 0.9991574287, 0.9954360127, 0.9..."
1,1,nerq7obwagPAy5Nso2xbkD,"Хорошее начало дня ,ментов всех сажают,супер ,...",telegram__egovpress,2026-05-16 22:05:08.728025,__label__rus_Cyrl,0.999572,"[Хорошее, начало, дня, ,, ментов, всех, сажают...","[[0, 7], [8, 14], [15, 18], [19, 20], [20, 26]...","[Хорошее, начало, дня, ,, ментов, всех, сажают...","[5, 5, 5, 7, 5, 5, 5, 7, 0, 7, 0, 5, 5, 5, 7, ...","[0.9990440011, 0.9983866215, 0.9984891415, 0.9..."
2,2,gAzFbhgY553A3Vjw36dT3d,Что-то как-то мало... В Алматы от миллиона ставка,telegram__egovpress,2026-05-16 22:05:08.728025,__label__rus_Cyrl,0.993228,"[Что, -, то, как, -, то, мало, ., ., ., В, Алм...","[[0, 3], [3, 4], [4, 6], [7, 10], [10, 11], [1...","[Что, -, то, как, -, то, мало, ., ., ., В, Алм...","[5, 7, 5, 5, 7, 5, 5, 7, 7, 7, 5, 5, 5, 5, 5]","[0.9989432693, 0.9993928671000001, 0.999037861..."
3,3,SMVMyhmDwRTZfDj3KYPfh5,"Патрули коррупция,пачкой сажать смело гой",telegram__egovpress,2026-05-16 22:05:08.728025,__label__rus_Cyrl,0.992829,"[Патрули, коррупция, ,, пачкой, сажать, смело,...","[[0, 7], [8, 17], [17, 18], [18, 24], [25, 31]...","[Патрули, коррупция, ,, пачкой, сажать, смело,...","[5, 0, 7, 5, 5, 5, 5]","[0.9127165675000001, 0.8891833425000001, 0.999..."
4,4,5yNeuE46DSgS7TLa5Dcw8h,Автобусами приехать и задержать всех хапкой,telegram__egovpress,2026-05-16 22:05:08.728025,__label__rus_Cyrl,0.998749,"[Автобусами, приехать, и, задержать, всех, хап...","[[0, 10], [11, 19], [20, 21], [22, 31], [32, 3...","[Автобусами, приехать, и, задержать, всех, хап...","[5, 5, 5, 5, 5, 5]","[0.9973257780000001, 0.9986483455, 0.999014258..."
...,...,...,...,...,...,...,...,...,...,...,...,...
1009154,1122786,jd3Kz8g7eHFa4LXDMRvSMB,Сәлеметсіз бе! Егер SPF-ті күнделікті 1 рет ви...,youtube__zhanerke_seyfullina,2026-05-16 22:09:33.209820,__label__kaz_Cyrl,1.000009,"[Сәлеметсіз, бе, !, Егер, SPF, -, ті, күнделік...","[[0, 10], [11, 13], [13, 14], [15, 19], [20, 2...","[Сәлеметсіз, бе, !, Егер, SPF, -, ті, күнделік...","[1, 1, 7, 1, 4, 7, 1, 1, 7, 1, 1, 1, 1, 1, 1, ...","[0.9990519881000001, 0.9993767142000001, 0.999..."
1009155,1122787,VEJf8oUEi6jdU2yojTc288,"Барлығы керемеет, түсінікті [EMOJI]",youtube__zhanerke_seyfullina,2026-05-16 22:09:33.209820,__label__kaz_Cyrl,1.000010,"[Барлығы, керемеет, ,, түсінікті, [EMOJI]]","[[0, 7], [8, 16], [16, 17], [18, 27], [28, 35]]","[Барлығы, керемеет, ,, түсінікті, [EMOJI]]","[1, 1, 7, 1, 7]","[0.998600781, 0.9978037477, 0.9988923669, 0.99..."
1009156,1122788,DmbcSTtzTrCr9A5hHGPRiw,Сәлеметсіз бе! Қалыңыз қалай? мүмкін бір айға,youtube__zhanerke_seyfullina,2026-05-16 22:09:33.209820,__label__kaz_Cyrl,1.000010,"[Сәлеметсіз, бе, !, Қалыңыз, қалай, ?, мүмкін,...","[[0, 10], [11, 13], [13, 14], [15, 22], [23, 2...","[Сәлеметсіз, бе, !, Қалыңыз, қалай, ?, мүмкін,...","[1, 1, 7, 1, 1, 7, 1, 1, 1]","[0.9979218841, 0.999311924, 0.9994689822, 0.99..."
1009157,1122789,Eyik3YZ5ZwwgRXmLAPvzQP,Сіз Алматыға келген қызсыз ғой иә?,youtube__zhanerke_seyfullina,2026-05-16 22:09:33.209820,__label__kaz_Cyrl,1.000008,"[Сіз, Алматыға, келген, қызсыз, ғой, иә, ?]","[[0, 3], [4, 12], [13, 19], [20, 26], [27, 30]...","[Сіз, Алматыға, келген, қызсыз, ғой, иә, ?]","[1, 1, 1, 1, 1, 1, 7]","[0.9984482527, 0.9524078369000001, 0.999200999..."


In [1]:
label2id = model.config.label2id
id2label = model.config.id2label
# or use the dictionaries from shared.py

In [4]:
predicted_data['unique_tags'] = predicted_data['labels'].apply(lambda l: set(l))

In [6]:
def if_csw(tags):
    # kz, ru
    if set((1,5)).issubset(tags):
        return True
    # skz, ru
    if set((6,5)).issubset(tags):
        return True
    # mixed_kz-ru
    if set((2,)).issubset(tags):
        return True
    # mixed_ru-kz
    if set((3,)).issubset(tags):
        return True
    return False

predicted_data['if_csw'] = predicted_data['unique_tags'].apply(if_csw)

In [8]:
predicted_data['if_csw'].value_counts() / len(predicted_data)

if_csw
False    0.920011
True     0.079989
Name: count, dtype: float64

In [9]:
predicted_data = predicted_data[(predicted_data['if_csw']==True)]
predicted_data.info()

<class 'pandas.DataFrame'>
Index: 80722 entries, 7 to 1009142
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          80722 non-null  int64  
 1   uuid                80722 non-null  str    
 2   clean_comment_text  80722 non-null  str    
 3   source              80722 non-null  str    
 4   ts                  80722 non-null  str    
 5   label               80722 non-null  str    
 6   __proba             80722 non-null  float64
 7   tokens              80722 non-null  object 
 8   spans               80722 non-null  object 
 9   words               80722 non-null  object 
 10  labels              80722 non-null  object 
 11  probas              80722 non-null  object 
 12  unique_tags         80722 non-null  object 
 13  if_csw              80722 non-null  bool   
dtypes: bool(1), float64(1), int64(1), object(6), str(5)
memory usage: 8.7+ MB


In [29]:
predicted_data.drop(predicted_data[predicted_data['tokens_'].apply(len)!=predicted_data['words'].apply(len)].index, inplace=True)

In [39]:
def convert_id2label(lst):
    """
    Превращаем id тега в строку
    """

    return [id2label.get(t, 'UNK') for t in lst]

predicted_data['labels_str'] = predicted_data['labels'].apply(convert_id2label)

In [48]:
predicted_data.to_json('data/main_corpus.jsonl', orient='records', lines=True)